# Finance Analysis of Tech Stocks Using yfinance

**Objective:**  
In this notebook, we analyze a publicly-traded tech stock (Microsoft, ticker "MSFT") by examining its Price-to-Earnings (PE) ratio over the most recent years and correlating it with its earned revenue. We then compare MSFT with five other tech stocks (Apple, Alphabet, Amazon, Meta, and Nvidia). Our analysis consists of the following steps:

1. **MSFT Analysis:**  
   - **Data Retrieval:**  
     We fetch annual earnings and revenue data using current yfinance income statement fields when available, with SEC Company Facts as a fallback for GitHub Actions runs. From the financials, we extract the **Total Revenue** and **Net Income** as proxies for revenue and earnings respectively.  
   - **Price Data:**  
     We download historical price data and resample it to obtain year-end closing prices using the recommended `'YE'` alias. If Yahoo rate-limits yfinance, the notebook falls back to Stooq through pandas_datareader.  
   - **Calculations:**  
     Using the current shares outstanding (from `ticker.info`), we compute a simplified Earnings Per Share (EPS) and derive the PE ratio for each available year.
   - **Visualization:**  
     A dual–axis line chart displays the evolution of MSFT’s PE ratio (left axis) and its annual revenue (right axis). This visualization serves as the basis for discussing whether there is a correlation between MSFT’s valuation (PE ratio) and its revenue performance.

2. **Peer Comparison:**  
   - The same data retrieval and computation steps are applied to five additional tech stocks: Apple (AAPL), Alphabet (GOOGL), Amazon (AMZN), Meta (META), and Nvidia (NVDA).
   - We align the data by computing the intersection of the available years across all stocks. (Due to missing data in earlier years, the common years are currently [2022, 2023, 2024].)
   - A multi–series line chart is plotted to show the annual PE ratios of all six stocks. Additionally, we calculate and plot the average PE ratio of the peer group compared to MSFT.
   - A five–number summary (boxplot) of the closing prices over the last 90 days is also generated.

3. **Volatility and COVID Impact:**  
   - We analyze recent market volatility using a boxplot of the last 90 days’ closing prices.
   - We also extract price data for Q1 and Q2 of 2020 to investigate the impact of COVID-19 on the tech sector. A line chart is used to display the price trends during this period.

4. **Revenue Distribution:**  
   - Pie charts are created for each stock to illustrate the relative distribution of annual revenue. Before plotting, any years with missing revenue data (NaN values) are dropped to ensure a correct visualization.

---

## Additional Notes

- **Earnings Data Deprecation:**  
  The notebook avoids deprecated `Ticker.earnings` usage. It uses `income_stmt` / `financials` when available and falls back to SEC Company Facts for annual revenue and net income.

- **Resampling Frequency Warning:**  
  The notebook uses the recommended `'YE'` alias for year-end resampling.

- **Data Limitations:**  
  Because the fallback financial data covers a shorter period, our analysis of annual trends is limited to the years for which data exists across all stocks (currently [2022, 2023, 2024]). Future work may involve integrating additional data sources to extend this timeframe.

- **Handling Missing Values:**  
  Before visualizing revenue distributions (via pie charts), rows with NaN revenue values are dropped so that the visualizations accurately represent available data.


In [ ]:
# %%
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import requests
from pandas_datareader import data as pdr
from datetime import datetime

# Set Matplotlib to inline mode (if using a notebook)
%matplotlib inline


# Step 1: Analysis of Microsoft (MSFT)

In this section, we fetch annual earnings and revenue data for Microsoft (MSFT) using yfinance. The notebook avoids deprecated `Ticker.earnings` usage. It first uses current yfinance income statement data and falls back to SEC Company Facts when yfinance fundamentals are unavailable. From the financials, we extract **Total Revenue** and **Net Income** as proxies for revenue and earnings, respectively.

We also download historical price data and resample it to obtain year–end closing prices. (Note: Although our previous version used the alias `'A'` for annual resampling, we have now updated the code to use the recommended `'YE'` alias for year-end sampling.)

Using the current shares outstanding (from `ticker.info`), we compute a simplified Earnings Per Share (EPS) and derive the Price-to-Earnings (PE) ratio for each available year. Due to data limitations (with some years missing values), our analysis now focuses on the common years available across all stocks—in this study, these are [2022, 2023, 2024] rather than a full 8‑year period.


In [ ]:
def get_price_history(symbol, start=None, end=None, period=None):
    import io
    import requests
    import pandas as pd
    import yfinance as yf
    from pandas_datareader import data as pdr
    from datetime import datetime

    if end is None:
        end = datetime.today().strftime("%Y-%m-%d")

    if start is None:
        if period == "90d":
            # Ask for extra calendar days so there are about 90 trading days available.
            start = (pd.Timestamp.today() - pd.DateOffset(days=140)).strftime("%Y-%m-%d")
        else:
            start = "2018-01-01"

    def clean_price_df(df):
        if df is None or df.empty:
            return None

        df = df.copy()
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()

        # Normalize possible lowercase column names.
        rename_map = {
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "volume": "Volume",
        }
        df = df.rename(columns=rename_map)

        if "Close" not in df.columns:
            return None

        return df

    # Try yfinance first for local and Colab runs.
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=start, end=end)
        df = clean_price_df(df)

        if df is not None:
            return df

    except Exception as e:
        print(f"yfinance price history unavailable for {symbol}; using fallback data source. Reason: {e}")

    # Stooq direct CSV fallback. This avoids pandas_datareader's Stooq parser,
    # which can fail in GitHub Actions if Stooq returns a nonstandard response.
    stooq_symbol = f"{symbol.lower()}.us"
    d1 = pd.to_datetime(start).strftime("%Y%m%d")
    d2 = pd.to_datetime(end).strftime("%Y%m%d")
    stooq_url = f"https://stooq.com/q/d/l/?s={stooq_symbol}&d1={d1}&d2={d2}&i=d"

    try:
        headers = {
            "User-Agent": "Mozilla/5.0 python-finance-analytics",
            "Accept": "text/csv,text/plain,*/*",
        }
        response = requests.get(stooq_url, headers=headers, timeout=30)
        response.raise_for_status()

        text = response.text.strip()
        if not text or text.lower().startswith("<!doctype") or "<html" in text.lower():
            raise ValueError("Stooq returned HTML instead of CSV.")

        df = pd.read_csv(io.StringIO(text))

        if "Date" not in df.columns or "Close" not in df.columns:
            raise ValueError(f"Unexpected Stooq CSV columns: {list(df.columns)}")

        df["Date"] = pd.to_datetime(df["Date"])
        df = df.set_index("Date").sort_index()
        df = clean_price_df(df)

        if df is not None:
            return df

    except Exception as e:
        print(f"Direct Stooq CSV unavailable for {symbol}. Reason: {e}")

    # Last fallback: pandas_datareader Stooq. This may fail in some hosted runners,
    # but it is useful locally.
    try:
        df = pdr.DataReader(stooq_symbol, "stooq", start=start, end=end)
        df = clean_price_df(df)

        if df is not None:
            return df

    except Exception as e:
        print(f"pandas_datareader Stooq unavailable for {symbol}. Reason: {e}")

    raise ValueError(f"No price history available for {symbol} from yfinance or Stooq.")


def get_annual_financials(symbol):
    import time
    import requests
    import yfinance as yf
    import pandas as pd
    import numpy as np
    from datetime import datetime

    SEC_CIKS = {
        "MSFT": "0000789019",
        "AAPL": "0000320193",
        "GOOGL": "0001652044",
        "AMZN": "0001018724",
        "META": "0001326801",
        "NVDA": "0001045810",
        "INTC": "0000050863",
        "AMD": "0000002488",
        "CSCO": "0000858877",
        "ORCL": "0001341439",
        "IBM": "0000051143",
    }

    def get_from_yfinance(ticker_obj):
        try:
            fin_df = ticker_obj.income_stmt

            if fin_df is None or fin_df.empty:
                fin_df = ticker_obj.financials

            if fin_df is None or fin_df.empty:
                return None

            fin_df = fin_df.T
            fin_df.index = pd.to_datetime(fin_df.index)

            if "Total Revenue" not in fin_df.columns or "Net Income" not in fin_df.columns:
                return None

            df = fin_df[["Total Revenue", "Net Income"]].rename(
                columns={
                    "Total Revenue": "Revenue",
                    "Net Income": "Earnings"
                }
            )

            df["Year"] = df.index.year
            df = df.reset_index(drop=True)
            return df[["Year", "Revenue", "Earnings"]]

        except Exception as e:
            print(f"yfinance financial statements unavailable for {symbol}. Reason: {e}")
            return None

    def get_fact_series(facts, tag_candidates, units_preference=("USD", "shares")):
        us_gaap = facts.get("facts", {}).get("us-gaap", {})

        for tag in tag_candidates:
            if tag not in us_gaap:
                continue

            units = us_gaap[tag].get("units", {})

            for unit_name in units_preference:
                values = units.get(unit_name, [])

                annual_rows = []
                for item in values:
                    if item.get("form") != "10-K":
                        continue

                    year = item.get("fy")
                    value = item.get("val")

                    if year is None or value is None:
                        continue

                    annual_rows.append({
                        "Year": int(year),
                        "Value": float(value),
                        "Filed": item.get("filed", "")
                    })

                if annual_rows:
                    out = pd.DataFrame(annual_rows)

                    # Keep the most recently filed value per fiscal year.
                    out = (
                        out.sort_values(["Year", "Filed"])
                           .drop_duplicates(subset=["Year"], keep="last")
                           .sort_values("Year")
                    )

                    return out[["Year", "Value"]]

        return pd.DataFrame(columns=["Year", "Value"])

    def get_from_sec(symbol):
        cik = SEC_CIKS.get(symbol.upper())

        if cik is None:
            return None

        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

        headers = {
            "User-Agent": "Kyle Barnett python-finance-analytics kbarnett0306@gmail.com",
            "Accept-Encoding": "gzip, deflate",
            "Host": "data.sec.gov",
        }

        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            facts = response.json()
        except Exception as e:
            print(f"SEC Company Facts unavailable for {symbol}. Reason: {e}")
            return None

        revenue = get_fact_series(
            facts,
            [
                "RevenueFromContractWithCustomerExcludingAssessedTax",
                "Revenues",
                "SalesRevenueNet",
            ],
            units_preference=("USD",)
        ).rename(columns={"Value": "Revenue"})

        earnings = get_fact_series(
            facts,
            [
                "NetIncomeLoss",
                "ProfitLoss",
            ],
            units_preference=("USD",)
        ).rename(columns={"Value": "Earnings"})

        shares = get_fact_series(
            facts,
            [
                "WeightedAverageNumberOfDilutedSharesOutstanding",
                "WeightedAverageNumberOfSharesOutstandingDiluted",
                "WeightedAverageNumberOfSharesOutstandingBasic",
                "CommonStockSharesOutstanding",
            ],
            units_preference=("shares",)
        ).rename(columns={"Value": "Shares"})

        if revenue.empty or earnings.empty:
            return None

        df = pd.merge(revenue, earnings, on="Year", how="inner")

        if not shares.empty:
            df = pd.merge(df, shares, on="Year", how="left")

        df = df.sort_values("Year").reset_index(drop=True)

        return df

    ticker = yf.Ticker(symbol)

    earnings_df = get_from_yfinance(ticker)

    if earnings_df is None or earnings_df.empty:
        print(f"yfinance financial statements unavailable for {symbol}; using SEC Company Facts fallback.")
        earnings_df = get_from_sec(symbol)

    if earnings_df is None or earnings_df.empty:
        raise ValueError(f"No annual revenue / earnings data available for {symbol}.")

    start_year = int(earnings_df["Year"].min()) - 1
    start_date = f"{start_year}-01-01"
    end_date = datetime.today().strftime("%Y-%m-%d")

    price_df = get_price_history(symbol, start=start_date, end=end_date)

    if price_df is None or price_df.empty:
        raise ValueError(f"No price data available for {symbol}.")

    annual_price = price_df["Close"].resample("YE").last()
    annual_price.index = annual_price.index.year

    # Prefer annual diluted share counts from SEC when available. This avoids
    # yfinance info/fast_info rate-limit failures in GitHub Actions.
    shares_outstanding = None
    if "Shares" in earnings_df.columns and earnings_df["Shares"].notna().any():
        shares_outstanding = None
    else:
        try:
            shares_outstanding = ticker.info.get("sharesOutstanding")
        except Exception as e:
            print(f"ticker.info shares outstanding unavailable for {symbol}. Reason: {e}")

        if shares_outstanding is None or shares_outstanding <= 0:
            try:
                shares_outstanding = ticker.fast_info.get("shares")
            except Exception as e:
                print(f"ticker.fast_info shares unavailable for {symbol}. Reason: {e}")
                shares_outstanding = None

        if shares_outstanding is None or shares_outstanding <= 0:
            raise ValueError(f"Shares outstanding not found or invalid for {symbol}.")

    common_years = sorted(set(annual_price.index).intersection(set(earnings_df["Year"])))

    df = earnings_df[earnings_df["Year"].isin(common_years)].copy()
    df["Price"] = df["Year"].apply(lambda y: annual_price.loc[y])

    # Correct: revenue, earnings, and share counts are already in their reported units.
    if "Shares" in df.columns and df["Shares"].notna().any():
        df["EPS"] = df.apply(
            lambda row: row["Earnings"] / row["Shares"]
            if pd.notna(row.get("Shares")) and row.get("Shares") > 0
            else np.nan,
            axis=1
        )

        if df["EPS"].isna().any() and shares_outstanding is not None:
            df["EPS"] = df["EPS"].fillna(df["Earnings"] / shares_outstanding)
    else:
        df["EPS"] = df["Earnings"] / shares_outstanding

    df["PE"] = df.apply(
        lambda row: row["Price"] / row["EPS"] if row["EPS"] > 0 else np.nan,
        axis=1
    )

    if "Shares" not in df.columns:
        df["Shares"] = np.nan

    df = df[["Year", "Revenue", "Earnings", "Shares", "Price", "EPS", "PE"]]
    df = df.sort_values("Year").reset_index(drop=True)

    # Be polite to live data APIs during batch runs.
    time.sleep(0.2)

    return df

In [ ]:
# Fetch Microsoft’s annual financials into msft_df
msft_df = get_annual_financials("MSFT")
msft_df.head()

In [ ]:
# Q1: Correlation between MSFT’s PE Ratio and Revenue
import numpy as np

# Prepare numeric series and drop any missing
df_corr = msft_df[["Revenue", "PE"]].copy()
df_corr["Revenue"] = pd.to_numeric(df_corr["Revenue"], errors="coerce")
df_corr["PE"] = pd.to_numeric(df_corr["PE"], errors="coerce")
df_corr = df_corr.dropna()

x = df_corr["Revenue"].values
y = df_corr["PE"].values

# Scatter + best-fit line
plt.figure(figsize=(8, 5))
plt.scatter(x, y, marker="o", label="Data points")

if len(df_corr) >= 2:
    m, b = np.polyfit(x, y, 1)
    r_squared = np.corrcoef(x, y)[0, 1] ** 2
    plt.plot(x, m * x + b, color="red", label=f"Fit line (R² = {r_squared:.2f})")

plt.xlabel("Revenue (USD)")
plt.ylabel("PE Ratio")
plt.title("MSFT: PE Ratio vs. Revenue")
plt.legend()
plt.tight_layout()
plt.show()

if len(df_corr) >= 2:
    corr = np.corrcoef(x, y)[0, 1]
    print(f"Pearson correlation between Revenue and PE: {corr:.2f}")
else:
    print("Not enough data points to compute Pearson correlation.")


# Visualization: MSFT Revenue and PE Ratio

The following dual–axis line chart displays Microsoft (MSFT)'s annual Price-to-Earnings (PE) ratio alongside its corresponding revenue for the common available years ([2022, 2023, 2024]).

This visualization helps us evaluate whether there is any noticeable correlation between MSFT’s valuation (as represented by its PE ratio) and its revenue performance.

*Review the chart and consider how changes in revenue relate to shifts in the PE ratio, noting any direct or inverse trends.*


In [ ]:
# Create a dual-axis plot for PE Ratio and Revenue
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot PE Ratio on primary y-axis
ax1.plot(msft_df["Year"], msft_df["PE"], marker="o", label="PE Ratio", color="blue")
ax1.set_xlabel("Year")
ax1.set_ylabel("PE Ratio", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")

# Plot Revenue on secondary y-axis in billions USD
ax2 = ax1.twinx()
ax2.plot(
    msft_df["Year"],
    msft_df["Revenue"] / 1_000_000_000,
    marker="s",
    label="Revenue (Billions USD)",
    color="green"
)
ax2.set_ylabel("Revenue (Billions USD)", color="green")
ax2.tick_params(axis="y", labelcolor="green")

plt.title("MSFT: Annual PE Ratio and Revenue")
fig.tight_layout()
plt.show()


# Discussion – Question 1

Inspect the dual–axis visualization for MSFT's annual PE ratio and revenue. Consider the following:

- How do changes in the PE ratio align with the changes in revenue over the analyzed common years ([2022, 2023, 2024])?
- Do you observe any direct or inverse trends between MSFT's valuation and its revenue performance?
- What other factors (e.g., market sentiment or earnings quality) might be influencing the PE ratio?

Based on these observations, discuss whether a clear correlation exists between MSFT's PE ratio and its revenue trend.


# Step 2: Comparison with Peer Tech Stocks

For the peer comparison, we apply the same data retrieval and calculation process used for MSFT to five additional tech stocks: Apple (AAPL), Alphabet (GOOGL), Amazon (AMZN), Meta (META), and Nvidia (NVDA).

After obtaining the annual financial data for each stock, we align the available years by taking the intersection of the datasets. Because of missing data in some earlier years (for example, 2020), the common years used in our analysis have been narrowed to [2022, 2023, 2024].

A multi–series line chart is then plotted to display the annual PE ratios for all six stocks. Additionally, we compute and plot the average PE ratio of the peer group alongside MSFT’s value, providing insight into MSFT’s relative valuation within the tech sector.


In [ ]:
# %%
# Define list of peer tickers
peer_tickers = ["AAPL", "GOOGL", "AMZN", "META", "NVDA"]

# Create a dictionary to store each stock's annual financial DataFrame
peer_data = {}
for symbol in peer_tickers:
    try:
        df = get_annual_financials(symbol)
        peer_data[symbol] = df
        print(f"{symbol} data:")
        print(df)
    except Exception as e:
        print(f"Error processing {symbol}: {e}")

# Align the years across stocks.
# We find the intersection of available years across all DataFrames (including MSFT)
years_set = set(msft_df['Year'])
for sym, df in peer_data.items():
    years_set = years_set.intersection(set(df['Year']))
common_years = sorted(list(years_set))

print("\nCommon Years across all stocks:", common_years)


In [ ]:
# %%
# Prepare a DataFrame for plotting PE ratios: rows are years, columns are ticker symbols
pe_data = pd.DataFrame({'Year': common_years})
pe_data.set_index('Year', inplace=True)

# Add MSFT first
msft_common = msft_df[msft_df['Year'].isin(common_years)].set_index('Year')
pe_data['MSFT'] = msft_common['PE']

# Now add each peer
for symbol, df in peer_data.items():
    df_common = df[df['Year'].isin(common_years)].set_index('Year')
    pe_data[symbol] = df_common['PE']

pe_data.sort_index(inplace=True)
print("Combined PE Ratio Data:")
print(pe_data)


# Visualization: Annual PE Ratios for All Stocks

The following line chart displays the annual PE ratios for Microsoft (MSFT) and its five peer tech stocks over the common available years ([2022, 2023, 2024]).

**Discussion Points:**
- Compare the trend for MSFT with the average PE ratio computed across the peer stocks.
- Does MSFT appear over– or under–valued relative to its peers based on these numbers?
- What might these trends indicate about investor sentiment or market dynamics within the tech sector?

Review the chart carefully and use these prompts to guide your analysis and discussion.


# Discussion – Question 2

Review the multi–series line chart that compares the annual PE ratios of MSFT and its peer group, as well as the chart showing MSFT vs. the peer average. Consider the following:

- How does MSFT’s annual PE ratio compare to the average PE ratio of its peer stocks?
- Does MSFT appear over–valued or under–valued relative to the sector average based on the trends you see?
- What might these comparisons indicate about investor sentiment or the company's performance relative to the industry?

Discuss your interpretation of MSFT’s relative valuation using the visualizations.



In [ ]:
# %%
# Plot a multi-series line chart for the PE ratios
plt.figure(figsize=(10, 6))
for symbol in pe_data.columns:
    plt.plot(pe_data.index, pe_data[symbol], marker='o', label=symbol)
plt.xlabel("Year")
plt.ylabel("PE Ratio")
plt.title("Annual PE Ratios: MSFT vs. Peer Tech Stocks")
plt.legend()
plt.tight_layout()
plt.show()

# Compute the average PE ratio of the peer group for each common year
pe_data['Peer Average'] = pe_data[peer_tickers].mean(axis=1)
plt.figure(figsize=(10, 6))
plt.plot(pe_data.index, pe_data['MSFT'], marker='o', label='MSFT')
plt.plot(pe_data.index, pe_data['Peer Average'], marker='s', label='Peer Average', linestyle='--')
plt.xlabel("Year")
plt.ylabel("PE Ratio")
plt.title("MSFT vs. Peer Average PE Ratio")
plt.legend()
plt.tight_layout()
plt.show()


# Step 3: Recent Stock Price Volatility & COVID Impact Analysis

In this section, we analyze recent market volatility and the impact of COVID-19 on the tech sector. Specifically, we:

- Retrieve and visualize the last 90 days of closing prices using a boxplot, which provides a five-number summary (minimum, first quartile, median, third quartile, and maximum) to illustrate recent market volatility.
- Extract daily closing prices for Q1 and Q2 of 2020 and plot a line chart to examine the price trends during the early phase of the COVID-19 outbreak.

These visualizations help us discuss whether the observed price movements support the conclusion that the tech sector was significantly affected by COVID-19.

*Discuss whether the data supports the conclusion that the selected industry was significantly affected by COVID based on the observed volatility and trends during Q1/Q2 2020.*


In [ ]:
# %%
import datetime

# Define all six ticker symbols (MSFT plus peers)
all_tickers = ["MSFT"] + peer_tickers

# Initialize a dictionary to store the last 90 days closing prices.
price_last90 = {}

# Loop through each ticker and download 90 days of history.
for symbol in all_tickers:
    df_hist = get_price_history(symbol, period="90d")
    price_last90[symbol] = df_hist["Close"].tail(90)

# Combine the series into a DataFrame
price_90_df = pd.DataFrame(price_last90)
price_90_df.head()


In [ ]:
# %%
# Create a boxplot for the last 90 days closing prices
plt.figure(figsize=(10, 6))
price_90_df.boxplot()
plt.ylabel("Closing Price (USD)")
plt.title("Five-Number Summary of Closing Prices (Last 90 Days)")
plt.tight_layout()
plt.show()


# COVID Impact Analysis

In this section, we investigate the impact of COVID-19 on the tech sector by analyzing historical price data during the early phase of the pandemic. Specifically, we extract and visualize daily closing prices for the first two quarters of 2020 (Q1 and Q2). This analysis is meant to capture the market reaction during the initial COVID outbreak.

Review the resulting line chart and discuss how the price trends during Q1/Q2 2020 compare with more recent market data, and whether these trends support the conclusion that the tech sector was significantly affected by COVID-19.


In [ ]:
# %%
# Define the start and end dates for Q1 and Q2 of 2020
start_date = "2020-01-01"
end_date = "2020-06-30"

# Dictionary to hold 2020 prices for all stocks
prices_2020 = {}

for symbol in all_tickers:
    df_2020 = get_price_history(symbol, start=start_date, end=end_date)
    prices_2020[symbol] = df_2020["Close"]

# Create a single DataFrame from these series
prices_2020_df = pd.DataFrame(prices_2020)

# Plot a line chart for the period
plt.figure(figsize=(10, 6))
for symbol in prices_2020_df.columns:
    plt.plot(prices_2020_df.index, prices_2020_df[symbol], label=symbol)
plt.xlabel("Date")
plt.ylabel("Closing Price (USD)")
plt.title("Stock Prices During Q1 and Q2 2020 (COVID Impact)")
plt.legend()
plt.tight_layout()
plt.show()


# Discussion – Question 3

Examine the visualizations for recent stock price volatility (the boxplot for the last 90 days) and the price trends during Q1/Q2 2020 (line chart). Consider these questions:

- Do the recent volatility patterns and the early COVID price trends suggest that the tech sector was significantly affected by COVID-19?
- How do the price movements during Q1/Q2 2020 compare to the more recent market data?
- What factors might be responsible for the observed volatility during the COVID period?

Based on the visual evidence, discuss whether the data supports the conclusion that the tech sector was significantly impacted by COVID-19.


# Step 4: Revenue Distribution Visualization

In this final analytical section, we visualize the relative distribution of annual revenue for each stock, based on the available common years ([2022, 2023, 2024]). For each stock, we use the retrieved earnings data to create a pie chart that shows the proportion of total revenue contributed by each available year. Before plotting, we drop any rows with missing revenue data (NaN values) to ensure that the visualizations accurately reflect the data.

*Interpret these pie charts in the context of revenue growth or changes over the available time periods.*


In [ ]:
def plot_revenue_pie(symbol, df):
    # Drop rows where Revenue is NaN
    df_clean = df.dropna(subset=['Revenue'])

    # Check if there is any data left to plot
    if df_clean.empty:
        print(f"No valid revenue data available for {symbol} to plot.")
        return

    # Use the years as labels and revenue values as sizes.
    labels = df_clean['Year'].astype(str)
    sizes = df_clean['Revenue']

    plt.figure(figsize=(6, 6))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    plt.title(f"{symbol} Revenue Distribution by Year")
    plt.tight_layout()
    plt.show()

# Conclusion and Additional Notes

**Discussion – Question 1:**  
The dual–axis visualization for MSFT allowed us to observe how its computed PE ratio and revenue evolve over time. Based on the trends, discuss whether a clear correlation exists between MSFT’s valuation (PE ratio) and its revenue performance. Consider other factors such as market sentiment and earnings quality in your discussion.

**Discussion – Question 2:**  
The comparison of MSFT’s annual PE ratio with that of its peer group (and the calculated peer average) offers insight into its relative valuation. Discuss whether MSFT appears over– or under–valued relative to the tech sector.

**Discussion – Question 3:**  
The analysis of recent volatility (boxplot of the last 90 days) combined with the price trends during Q1/Q2 2020 illustrates the impact of COVID-19 on the tech sector. Support your discussion of the COVID impact with observations from the provided charts.

---

### Additional Notes

- **Earnings Data Deprecation:**  
  The notebook avoids deprecated `Ticker.earnings` usage. It uses current yfinance income statement fields when available and falls back to SEC Company Facts for annual revenue, net income, and share counts.

- **Resampling Frequency Warning:**  
  The notebook uses the recommended `'YE'` alias for year-end resampling.

- **Data Limitations:**  
  Due to the limited historical coverage from the fallback financial data, our analysis is based on the common available years [2022, 2023, 2024]. Future work may involve integrating additional data sources to extend the analysis period.

- **Handling Missing Values:**  
  Before visualizing revenue distributions (via pie charts), we drop rows with missing revenue values to ensure accurate representations.


# Extended Analysis: Inflation, Model Robustness, and Sector Factors

**New Questions:**

1. **Q1 (Microsoft):** Correlation between monthly inflation and Microsoft’s earnings over the past 36 months (linear regression), and effect of doubling training sample size.  
2. **Q2 (Sector PE):** For five additional tech stocks, compute average annual PE over the past 8 years and test whether average PE is affected by inflation and consumer confidence (multiple linear regression), with a confusion matrix for directional accuracy.  
3. **Q3 (Hypothesis Testing):** Select another macro variable (unemployment rate) and test its correlation with sector average PE.  
4. **Q4 (Forecasting):** Compare models (LinearRegression, PolynomialRegression, RandomForest) in fitting MSFT closing price, plot actual vs predictions, and forecast for June 1, 2025.



In [ ]:
# %%
# Import additional libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import requests
from pandas_datareader import data as pdr
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, confusion_matrix

%matplotlib inline


# Question 1: Microsoft Earnings vs. Inflation

We compute monthly net income for MSFT and monthly inflation (CPI), then fit a simple linear regression to test for correlation. We also examine the effect of doubling the training sample size.


In [ ]:
# %%
# Utility functions for Question 1
def get_monthly_net_income(symbol, months=36):
    import pandas as pd
    import yfinance as yf

    ticker = yf.Ticker(symbol)

    try:
        qf = ticker.quarterly_financials.T

        if qf is not None and not qf.empty and "Net Income" in qf.columns:
            qf.index = pd.to_datetime(qf.index)

            # Forward-fill to month-end frequency, then evenly allocate quarterly income across those months.
            monthly = (
                qf[["Net Income"]]
                .resample("ME")
                .ffill()
                / 3.0
            )

            return monthly["Net Income"].tail(months)

    except Exception as e:
        print(f"yfinance quarterly financials unavailable for {symbol}; using annual fallback. Reason: {e}")

    # Robust fallback for GitHub Actions:
    # Use available annual earnings and spread each annual value across 12 months.
    annual = get_annual_financials(symbol)[["Year", "Earnings"]].dropna().sort_values("Year")

    monthly_rows = []
    for _, row in annual.iterrows():
        for month in range(1, 13):
            monthly_rows.append({
                "Date": pd.Timestamp(int(row["Year"]), month, 1) + pd.offsets.MonthEnd(0),
                "Net Income": float(row["Earnings"]) / 12.0
            })

    monthly = pd.DataFrame(monthly_rows).set_index("Date")["Net Income"]
    return monthly.tail(months)


def get_monthly_inflation(months=36):
    import pandas as pd
    from pandas_datareader import data as pdr
    from datetime import datetime

    # Fetch CPI from FRED and compute % change.
    cpi = pdr.DataReader(
        "CPIAUCSL",
        "fred",
        start=datetime.now() - pd.DateOffset(months=months + 2)
    )

    inf = cpi.pct_change().dropna().rename(columns={"CPIAUCSL": "Inflation"})
    inf.index = inf.index.to_period("M").to_timestamp("M")

    return inf["Inflation"].tail(months)


In [ ]:
# %%
# Fetch and merge data
msft_earn = get_monthly_net_income("MSFT", 36)
inflation = get_monthly_inflation(36)

df_q1 = (
    pd.concat([msft_earn, inflation], axis=1)
      .dropna()
      .rename(columns={"Net Income": "Earnings"})
)

print("df_q1.shape:", df_q1.shape)

# Keep X as a DataFrame so scikit-learn retains feature names.
X = df_q1[["Inflation"]]
y = df_q1["Earnings"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    shuffle=False,
    test_size=0.3
)

lr1 = LinearRegression().fit(X_train, y_train)
y_pred = lr1.predict(X_test)

# Plot regression
plt.figure(figsize=(8, 5))
plt.scatter(df_q1["Inflation"], df_q1["Earnings"], label="Data points")

xs = pd.DataFrame({
    "Inflation": np.linspace(df_q1["Inflation"].min(), df_q1["Inflation"].max(), 100)
})

plt.plot(
    xs["Inflation"],
    lr1.predict(xs),
    color="red",
    label=f"Fit: R²={r2_score(y_test, y_pred):.2f}"
)

plt.xlabel("Monthly Inflation Rate")
plt.ylabel("Monthly Net Income (USD)")
plt.title("MSFT Earnings vs. Inflation (36 months)")
plt.legend()
plt.tight_layout()
plt.show()

print("Out-of-sample R²:", r2_score(y_test, y_pred))

# Compare against a larger training sample.
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X,
    y,
    shuffle=False,
    test_size=0.15
)

lr1_large = LinearRegression().fit(X_train_large, y_train_large)
y_pred_large = lr1_large.predict(X_test_large)

print("Original training sample size:", len(X_train))
print("Larger training sample size:", len(X_train_large))
print("Original out-of-sample R²:", r2_score(y_test, y_pred))
print("Larger-training-sample out-of-sample R²:", r2_score(y_test_large, y_pred_large))


# Question 2: Sector PE vs. Inflation & Consumer Confidence

Select five additional tech stocks and compute their average annual PE over the past 8 years. Then test whether the sector’s average PE is influenced by inflation and consumer confidence via multiple linear regression, and evaluate directional accuracy with a confusion matrix.


In [ ]:
# %%
# Compute 8‑year PE history for sector tickers
sector_tickers = ["INTC","AMD","CSCO","ORCL","IBM"]  # adjust as needed
pe_df = pd.DataFrame({
    t: get_annual_financials(t).set_index('Year')['PE'].tail(8)
    for t in sector_tickers
})
pe_df['Average_PE'] = pe_df.mean(axis=1)
pe_df.index.name = 'Year'
pe_df


In [ ]:
# %%
# Merge macro data (year-end % changes) and align to sector PE years

# Annual inflation (% change at year-end)
inf_ann = (
    pdr.DataReader("CPIAUCSL", "fred", start=int(pe_df.index.min()))
       .resample("YE").last()
       .pct_change().dropna()
       .rename(columns={"CPIAUCSL": "Inflation"})
)
inf_ann.index = inf_ann.index.year

# Annual consumer confidence (% change at year-end)
cconf = (
    pdr.DataReader("UMCSENT", "fred", start=int(pe_df.index.min()))
       .resample("YE").last()
       .pct_change().dropna()
       .rename(columns={"UMCSENT": "Cons_Conf"})
)
cconf.index = cconf.index.year

# Combine macro series and keep only years present in pe_df
macro = pd.concat([inf_ann, cconf], axis=1).dropna()
common_years = macro.index.intersection(pe_df.index)
macro = macro.loc[common_years]

# Slice sector average PE to those same years
pe_sub = pe_df["Average_PE"].loc[common_years]

# Build the regression DataFrame and drop incomplete years.
df_q2 = pd.concat([pe_sub, macro], axis=1).dropna()

if len(df_q2) >= 3:
    X2 = df_q2[["Inflation", "Cons_Conf"]]
    y2 = df_q2["Average_PE"]

    X2tr, X2te, y2tr, y2te = train_test_split(
        X2,
        y2,
        test_size=0.25,
        shuffle=False
    )

    mlr = LinearRegression().fit(X2tr, y2tr)
    y2pred = mlr.predict(X2te)

    # Plot actual vs. predicted average PE.
    plt.figure(figsize=(8, 5))
    plt.plot(df_q2.index, y2, marker="o", label="Actual Avg PE")
    plt.plot(
        y2te.index,
        y2pred,
        marker="x",
        linestyle="--",
        label="Predicted Avg PE"
    )
    plt.xlabel("Year")
    plt.ylabel("PE Ratio")
    plt.title("Sector Avg PE vs. Inflation & Consumer Confidence")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Regression Coefficients:", dict(zip(["Inflation", "Cons_Conf"], mlr.coef_)))
    print("Intercept:", mlr.intercept_)

    # Confusion matrix for directional accuracy requires at least two test observations.
    if len(y2te) >= 2:
        y2te_up = (y2te.values[1:] > y2te.values[:-1]).astype(int)
        y2pred_up = (y2pred[1:] > y2pred[:-1]).astype(int)
        cm = confusion_matrix(y2te_up, y2pred_up, labels=[0, 1])
        print("Confusion Matrix (up/down):\n", cm)
    else:
        print("Not enough test observations to compute a directional confusion matrix.")
else:
    print("Not enough aligned annual observations to run the multiple linear regression.")


# Question 3: Hypothesis Test with Unemployment

Test whether changes in the unemployment rate correlate with the sector’s average PE. We perform a simple linear regression with annual ΔUnemployment as predictor.


In [ ]:
# %%
# Question 3: Hypothesis Test with Unemployment

# Annual unemployment rate change (year-end % change)
unemp = (
    pdr.DataReader("UNRATE", "fred", start=int(pe_df.index.min()))
       .resample("YE").last()
       .pct_change()
       .dropna()
       .rename(columns={"UNRATE": "Unemployment"})
)
unemp.index = unemp.index.year

# Combine inflation, consumer confidence, and unemployment into macro2
macro2 = pd.concat([inf_ann, cconf, unemp], axis=1).dropna()

# Restrict to years present in our sector PE DataFrame
common_years2 = macro2.index.intersection(pe_df.index)
macro2 = macro2.loc[common_years2]

# Slice sector average PE to those same years
pe_sub2 = pe_df["Average_PE"].loc[common_years2]

# Build regression DataFrame and drop incomplete years.
df_q3 = pd.concat([pe_sub2, macro2], axis=1).dropna()

if len(df_q3) >= 2:
    # Simple linear regression: Avg_PE ~ Unemployment
    X3 = df_q3[["Unemployment"]]
    y3 = df_q3["Average_PE"]

    lr3 = LinearRegression().fit(X3, y3)

    # Print model metrics
    print("Unemployment coefficient:", lr3.coef_[0])
    print("Intercept:", lr3.intercept_)
    print("R²:", lr3.score(X3, y3))

    # Plot the relationship
    plt.figure(figsize=(6, 4))
    plt.scatter(df_q3["Unemployment"], y3, label="Data")
    plt.plot(df_q3["Unemployment"], lr3.predict(X3), color="red", label="Fit")
    plt.xlabel("Annual % Change in Unemployment")
    plt.ylabel("Sector Average PE")
    plt.title("Sector Avg PE vs. Δ Unemployment")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Not enough aligned observations to run the unemployment regression.")


# Question 4: Model Comparison on MSFT Price

Compare LinearRegression, PolynomialRegression (degree 3), and RandomForest in fitting MSFT closing prices. Plot actual vs. predictions, and forecast the closing price for June 1, 2025.


In [ ]:
# %%
# Fetch MSFT history and prepare features
msft_hist = get_price_history(
    "MSFT",
    start="2018-01-01",
    end=datetime.today().strftime("%Y-%m-%d")
)

msft_hist["DateOrd"] = msft_hist.index.map(datetime.toordinal)
X4 = msft_hist[["DateOrd"]]
y4 = msft_hist["Close"]

# Train/test split
X4tr, X4te, y4tr, y4te = train_test_split(
    X4,
    y4,
    test_size=0.2,
    shuffle=False
)

# Linear Regression
lr4 = LinearRegression().fit(X4tr, y4tr)
pred_lr = lr4.predict(X4te)

# Polynomial Regression (degree=3)
pf = PolynomialFeatures(3)
Xp4 = pf.fit_transform(X4tr)
Xp4te = pf.transform(X4te)
pr = LinearRegression().fit(Xp4, y4tr)
pred_pr = pr.predict(Xp4te)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=0).fit(X4tr, y4tr)
pred_rf = rf.predict(X4te)

# Plot actual vs. models
plt.figure(figsize=(10, 5))
plt.plot(msft_hist.index, y4, label="Actual", alpha=0.6)
plt.plot(X4te.index, pred_lr, label="Linear", linewidth=2)
plt.plot(X4te.index, pred_pr, label="Poly(3)", linewidth=2)
plt.plot(X4te.index, pred_rf, label="RandomForest", linewidth=2)
plt.legend()
plt.title("MSFT Closing Price: Model Comparison")
plt.tight_layout()
plt.show()

# Forecast for June 1, 2025
future_date = pd.DataFrame({
    "DateOrd": [datetime(2025, 6, 1).toordinal()]
})

print("Predictions for 2025-06-01:")
print(" LinearReg:   ", lr4.predict(future_date)[0])
print(" Poly(3):     ", pr.predict(pf.transform(future_date))[0])
print(" RandomForest:", rf.predict(future_date)[0])


In [ ]:
# %%
# Q3 (pie charts): Revenue distribution by year for all 6 stocks
for sym, df in [('MSFT', msft_df)] + [(t, peer_data[t]) for t in peer_tickers]:
    plot_revenue_pie(sym, df)

# Conclusion

- **Q1:** Interpret the R² and slope from the MSFT earnings vs. inflation regression.  
- **Q1‑b:** Compare R² when increasing training size from 50 % to 75 %.  
- **Q2:** Examine regression coefficients on inflation and consumer confidence, and review the up/down confusion matrix.  
- **Q3:** Assess whether ΔUnemployment significantly predicts sector average PE.  
- **Q4:** Compare the three model fits visually and discuss each model’s forecast for June 1, 2025.
